In [15]:
from tensorflow import keras
from keras import layers
from keras_tuner.tuners import RandomSearch

In [27]:
import pandas as pd
import numpy as np

In [17]:
data = pd.read_csv("/content/Real_Combine.csv")

In [18]:
data.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [19]:
X = data.iloc[:,:-1]
y = data.iloc[:,-1]

# Hyperparameters

How many Hidden Layers should we have ?

How many number of neurons we should have in hidden layers ?

Learning Rate

In [39]:
def build(hp):

  model = keras.Sequential()
  for i in range(hp.Int('num_layers',2,20)):
    model.add(layers.Dense(units=hp.Int('units_'+str(i),min_value=32,max_value=512,step=32),activation='relu'))

  model.add(layers.Dense(1,activation = 'linear'))

  model.compile(optimizer = keras.optimizers.Adam(hp.Choice('learning_rate',[1e-2,1e-3,1e-4])),loss = 'mean_absolute_error',metrics = [keras.metrics.MeanAbsoluteError(name='mean_absolute_error')])

  return model

In [29]:
tuner = RandomSearch(

                     build,
                     objective = 'val_mean_absolute_error',
                     directory = 'project',
                     project_name = 'Air Quality Index',

                     max_trials= 5,
                     executions_per_trial = 3
)

In [30]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.3,random_state = 0)

The `loss` and `mean_absolute_error` are `nan` during training, and the tuner failed due to `RuntimeError: Number of consecutive failures exceeded the limit of 3`. This often happens when the input data contains `NaN` or infinite values, which can cause numerical instability in the model.

Let's check for `NaN` and infinite values in `X_train`, `y_train`, `X_test`, and `y_test`.

In [33]:
print('Checking X_train for NaNs:', X_train.isnull().sum().sum())
print('Checking y_train for NaNs:', y_train.isnull().sum())
print('Checking X_test for NaNs:', X_test.isnull().sum().sum())
print('Checking y_test for NaNs:', y_test.isnull().sum())

print('\nChecking X_train for inf values:', np.isinf(X_train).sum().sum())
print('Checking y_train for inf values:', np.isinf(y_train).sum())
print('Checking X_test for inf values:', np.isinf(X_test).sum().sum())
print('Checking y_test for inf values:', np.isinf(y_test).sum())

Checking X_train for NaNs: 0
Checking y_train for NaNs: 1
Checking X_test for NaNs: 0
Checking y_test for NaNs: 0

Checking X_train for inf values: 0
Checking y_train for inf values: 0
Checking X_test for inf values: 0
Checking y_test for inf values: 0


It looks like `y_train` has one NaN value. This is causing the model to output `nan` for loss and metrics. We need to remove the row with the NaN value from both `X_train` and `y_train` to maintain consistency.

In [34]:
nan_indices = y_train[y_train.isnull()].index
X_train = X_train.drop(nan_indices)
y_train = y_train.drop(nan_indices)

print('Checking y_train for NaNs after dropping:', y_train.isnull().sum())

Checking y_train for NaNs after dropping: 0


In [40]:
tuner.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 39ms/step - loss: 182.4228 - mean_absolute_error: 182.4228 - val_loss: 63.1839 - val_mean_absolute_error: 63.1839
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 87.5324 - mean_absolute_error: 87.5324 - val_loss: 71.4263 - val_mean_absolute_error: 71.4263
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 69.1693 - mean_absolute_error: 69.1693 - val_loss: 59.4708 - val_mean_absolute_error: 59.4708
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 64.7977 - mean_absolute_error: 64.7977 - val_loss: 67.6161 - val_mean_absolute_error: 67.6161
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 70.9328 - mean_absolute_error: 70.9328 - val_loss: 55.3651 - val_mean_absolute_error: 55.3651
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - loss: 139.6720 - mean_absolute_error: 139.6720 - val_loss: 103.1882 - val_mean_absolute_error: 103.1882
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 112.4200 - mean_absolute

/usr/local/lib/python3.12/dist-packages/keras_tuner/src/engine/metrics_tracking.py:111: RuntimeWarning: All-NaN axis encountered
  np.nanmin(values) if self.direction == "min" else np.nanmax(values)


RuntimeError: Number of consecutive failures exceeded the limit of 3.
